# Dynamiqs solver benchmark suite

This notebook is an interactive dashboard for the benchmark suite in `benchmarks/dynamiqs_benchmarks`.
It is intended for local solver exploration, reproducible benchmark logs, and long-term regression tracking.

The notebook can run every benchmark case created in the suite and then produce accuracy/speed visualizations:

- solver leaderboard tables,
- runtime vs relative-error scatter plots,
- benchmark × solver heatmaps for runtime and error,
- per-benchmark Pareto frontiers that show accuracy/speed tradeoffs,
- skipped/failed solver diagnostics.

## Dynamiqs benchmark design notes

The suite follows the solver structure exposed by Dynamiqs:

- `dq.sesolve(H, psi0, tsave, ...)` for closed Schrödinger dynamics.
- `dq.mesolve(H, jump_ops, rho0, tsave, ...)` for Lindblad/master-equation dynamics.
- Deterministic methods from `dynamiqs.method`: `Tsit5`, `Dopri5`, `Dopri8`, `Kvaerno3`, `Kvaerno5`, `Euler`, `Rouchon1`, `Rouchon2`, `Rouchon3`, and `Expm`.
- Time-dependent operators through `dq.modulated(...)` and `dq.pwc(...)`, matching driven and piecewise-constant benchmark cases.
- Solver diagnostics through `result.infos`, from which the runner extracts step counts and accepted/rejected steps when available.

Reference strategies are case-specific: analytical observable references (if available), `Expm` for feasible constant Liouvillians, or using tight double-precision for high-order ODE/Rouchon.

In [ ]:
from __future__ import annotations

import csv
import math
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

# Make the notebook robust when opened from benchmark_example/ or repo root.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = (
    NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmarks.dynamiqs_benchmarks.cases import benchmark_cases
from benchmarks.dynamiqs_benchmarks.plot import plot_results
from benchmarks.dynamiqs_benchmarks.runner import parse_csv_list, run_suite

In [ ]:
# --- Configuration ---------------------------------------------------------
# Use 'smoke' for a quick interactive pass, 'standard' for meaningful local
# comparisons, and 'full' for larger scaling runs.
PROFILE = 'smoke'
PRECISION = 'double'

# Set either value to a comma-separated string, e.g. 'Tsit5,Dopri8,Expm' or
# 'driven_damped_oscillator_mesolve'. Use None to run the default suite.
METHOD_FILTER = None
CASE_FILTER = None

# Timings exclude first-call JAX compilation when WARMUP=True.
WARMUP = True

OUTPUT_DIR = REPO_ROOT / 'benchmark_results' / f'notebook_{PROFILE}'
RESULTS_CSV = OUTPUT_DIR / 'results.csv'

In [ ]:
def markdown_table(headers, rows):
    table = ['| ' + ' | '.join(headers) + ' |']
    table.append('| ' + ' | '.join(['---'] * len(headers)) + ' |')
    for row in rows:
        table.append('| ' + ' | '.join(str(item) for item in row) + ' |')
    return '\n'.join(table)


def load_results(path):
    with path.open() as f:
        rows = list(csv.DictReader(f))
    numeric_columns = {'runtime_s', 'nsteps', 'naccepted', 'nrejected', 'error'}
    for row in rows:
        for key in numeric_columns:
            try:
                row[key] = float(row[key])
            except (TypeError, ValueError):
                row[key] = math.nan
    return rows


def status_counts(rows):
    counts = defaultdict(int)
    for row in rows:
        counts[row['status']] += 1
    return dict(sorted(counts.items()))

In [ ]:
case_rows = []
for case in benchmark_cases(PROFILE):
    case_rows.append(
        [f'`{case.name}`', case.kind, ', '.join(case.tags), case.reference_strategy]
    )

display(
    Markdown(
        markdown_table(['Benchmark', 'API', 'Tags', 'Reference strategy'], case_rows)
    )
)

## Run the benchmark suite

The cell below runs all configured cases/methods and writes `results.csv`, `leaderboard.csv`, and `metadata.json`.
For a previously generated run, skip this cell and load `RESULTS_CSV` in the next section.

In [ ]:
rows = run_suite(
    output_dir=OUTPUT_DIR,
    profile=PROFILE,
    selected_cases=parse_csv_list([CASE_FILTER]) if CASE_FILTER else None,
    selected_methods=parse_csv_list([METHOD_FILTER]) if METHOD_FILTER else None,
    precision=PRECISION,
    warmup=WARMUP,
)

display(
    Markdown(
        f'Wrote `{RESULTS_CSV.relative_to(REPO_ROOT)}` with status counts: '
        f'`{status_counts(rows)}`.'
    )
)

In [ ]:
rows = load_results(RESULTS_CSV)
passed = [row for row in rows if row['status'] == 'pass']
failed = [row for row in rows if row['status'] == 'fail']
skipped = [row for row in rows if row['status'] == 'skip']

summary_rows = [[key, value] for key, value in status_counts(rows).items()]
display(Markdown(markdown_table(['Status', 'Rows'], summary_rows)))

In [ ]:
leaderboard_rows = []
for benchmark in sorted({row['benchmark'] for row in passed}):
    group = sorted(
        [row for row in passed if row['benchmark'] == benchmark],
        key=lambda row: (row['error'], row['runtime_s']),
    )
    for rank, row in enumerate(group[:5], start=1):
        leaderboard_rows.append(
            [
                benchmark,
                rank,
                row['solver'],
                f'{row["runtime_s"]:.3g}',
                f'{row["error"]:.3e}',
                f'{row["nsteps"]:.1f}' if not math.isnan(row['nsteps']) else 'n/a',
                row['reference_solver'],
            ]
        )

display(
    Markdown(
        markdown_table(
            [
                'Benchmark',
                'Rank',
                'Solver',
                'Runtime (s)',
                'Rel. error',
                'Steps',
                'Reference',
            ],
            leaderboard_rows,
        )
    )
)

## Accuracy vs speed

A good solver is usually near the lower-left of these log-log plots: faster runtime and lower relative error.

In [ ]:
def scatter_accuracy_speed(rows, title='Dynamiqs solver timing vs accuracy'):
    fig, ax = plt.subplots(figsize=(10, 6))
    solvers = sorted({row['solver'] for row in rows})
    colors = {
        solver: plt.get_cmap('tab10')(idx % 10) for idx, solver in enumerate(solvers)
    }
    markers = {'sesolve': 'o', 'mesolve': 's'}

    for solver in solvers:
        group = [row for row in rows if row['solver'] == solver]
        ax.scatter(
            [row['runtime_s'] for row in group],
            [max(row['error'], 1e-16) for row in group],
            label=solver,
            color=colors[solver],
            marker=markers.get(group[0]['kind'], 'o'),
            alpha=0.85,
            s=80,
            edgecolor='black',
            linewidth=0.4,
        )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Runtime after warmup (s)')
    ax.set_ylabel('Relative error vs reference')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(ncols=2, fontsize='small')
    fig.tight_layout()
    return fig, ax


scatter_accuracy_speed(passed)
plt.show()

## Benchmark × solver heatmaps

These heatmaps make solver regressions easy to spot across runs. Darker runtime cells are slower; darker error cells are less accurate.

In [ ]:
def pivot(rows, value_key):
    benchmarks = sorted({row['benchmark'] for row in rows})
    solvers = sorted({row['solver'] for row in rows})
    matrix = np.full((len(benchmarks), len(solvers)), np.nan)
    for row in rows:
        i = benchmarks.index(row['benchmark'])
        j = solvers.index(row['solver'])
        value = row[value_key]
        if not math.isnan(value):
            matrix[i, j] = value
    return benchmarks, solvers, matrix


def heatmap(rows, value_key, title, colorbar_label):
    benchmarks, solvers, matrix = pivot(rows, value_key)
    matrix = np.where(matrix > 0, matrix, np.nan)
    log_matrix = np.log10(matrix)

    fig, ax = plt.subplots(
        figsize=(max(8, len(solvers) * 0.7), max(4, len(benchmarks) * 0.45))
    )
    image = ax.imshow(log_matrix, aspect='auto', cmap='viridis')
    ax.set_xticks(np.arange(len(solvers)), labels=solvers, rotation=35, ha='right')
    ax.set_yticks(np.arange(len(benchmarks)), labels=benchmarks)
    ax.set_title(title)

    for i in range(len(benchmarks)):
        for j in range(len(solvers)):
            value = matrix[i, j]
            label = '—' if math.isnan(value) else f'{value:.1e}'
            ax.text(j, i, label, ha='center', va='center', color='white', fontsize=8)

    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label(colorbar_label)
    fig.tight_layout()
    return fig, ax


heatmap(passed, 'runtime_s', 'Runtime heatmap', 'log10(runtime seconds)')
plt.show()

heatmap(passed, 'error', 'Relative-error heatmap', 'log10(relative error)')
plt.show()

## Per-benchmark Pareto frontiers

A solver is Pareto-efficient within a benchmark if no other passing solver is both faster and more accurate.

In [ ]:
def pareto_front(group):
    ordered = sorted(group, key=lambda row: row['runtime_s'])
    frontier = []
    best_error = math.inf
    for row in ordered:
        if row['error'] < best_error:
            frontier.append(row)
            best_error = row['error']
    return frontier


benchmarks = sorted({row['benchmark'] for row in passed})
cols = 2
rows_needed = math.ceil(len(benchmarks) / cols)
fig, axes = plt.subplots(
    rows_needed, cols, figsize=(12, 4 * rows_needed), squeeze=False
)

for ax, benchmark in zip(axes.ravel(), benchmarks):
    group = [row for row in passed if row['benchmark'] == benchmark]
    frontier = pareto_front(group)
    ax.scatter(
        [row['runtime_s'] for row in group],
        [max(row['error'], 1e-16) for row in group],
        alpha=0.65,
    )
    ax.plot(
        [row['runtime_s'] for row in frontier],
        [max(row['error'], 1e-16) for row in frontier],
        color='tab:red',
        marker='o',
        label='Pareto frontier',
    )
    for row in frontier:
        ax.annotate(
            row['solver'], (row['runtime_s'], max(row['error'], 1e-16)), fontsize=8
        )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(benchmark)
    ax.set_xlabel('Runtime (s)')
    ax.set_ylabel('Relative error')
    ax.grid(True, which='both', alpha=0.25)

for ax in axes.ravel()[len(benchmarks) :]:
    ax.axis('off')

fig.tight_layout()
plt.show()

In [ ]:
plot_paths = plot_results(RESULTS_CSV, OUTPUT_DIR / 'plots')
display(
    Markdown(
        'Saved static plot files:\n'
        + '\n'.join(f'- `{path.relative_to(REPO_ROOT)}`' for path in plot_paths)
    )
)

## Solver failures and skips

Failures are part of the benchmark signal: they capture stability issues, tolerance problems, and unsupported method/problem combinations.
Skips document practical defaults. Explicit `--method` selections in the CLI can opt into slow implicit combinations.

In [ ]:
diagnostic_rows = []
for row in failed + skipped:
    diagnostic_rows.append(
        [
            row['benchmark'],
            row['solver'],
            row['status'],
            row['message'][:160] + ('…' if len(row['message']) > 160 else ''),
        ]
    )

if diagnostic_rows:
    display(
        Markdown(
            markdown_table(
                ['Benchmark', 'Solver', 'Status', 'Message'], diagnostic_rows
            )
        )
    )
else:
    display(Markdown('No failed or skipped rows in this run.'))

## Tips for doing a deeper benchmarks based on existing methods

- Switch `PROFILE` to `standard` or `full` and keep the generated CSVs per commit.
- Compare two commits by loading both `results.csv` files and plotting ratios of runtime/error.
- Add memory measurements or gradient checks as secondary metrics once the primary deterministic suite is stable.